# Setup


# Configure Access to Cloud Storage via Unity Catalog

Access Cloud Storage

In [0]:
%fs ls 'abfss://eccomerce-container@ecommerceextdatalake.dfs.core.windows.net/' 

path,name,size,modificationTime
abfss://eccomerce-container@ecommerceextdatalake.dfs.core.windows.net/bronze/,bronze/,0,0
abfss://eccomerce-container@ecommerceextdatalake.dfs.core.windows.net/gold/,gold/,0,0
abfss://eccomerce-container@ecommerceextdatalake.dfs.core.windows.net/landing/,landing/,0,0
abfss://eccomerce-container@ecommerceextdatalake.dfs.core.windows.net/silver/,silver/,0,0


### Create External Location

In [0]:
%sql
CREATE EXTERNAL LOCATION IF NOT EXISTS ecommerce_ext_dl_eccomerce_container
    URL 'abfss://eccomerce-container@ecommerceextdatalake.dfs.core.windows.net/'
    WITH (STORAGE CREDENTIAL ecommerce_ext_sc)
    COMMENT 'External location for ecommerce data'

### Create Catalog - ecommerce

In [0]:
%sql
SHOW CATALOGS;

catalog
db_workspace_srvrls
ecommerce
samples
system


In [0]:
%sql
CREATE CATALOG IF NOT EXISTS ecommerce
      MANAGED LOCATION 'abfss://eccomerce-container@ecommerceextdatalake.dfs.core.windows.net/' 
      COMMENT 'Catalog for ecommerce data';     

%md
### Create Schema
1. landing
2. Bronze
3. Silver
4. Gold

In [0]:
%sql
SELECT current_catalog();

current_catalog()
ecommerce


In [0]:
%sql
USE CATALOG ecommerce;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS landing
    MANAGED LOCATION 'abfss://eccomerce-container@ecommerceextdatalake.dfs.core.windows.net/landing';

CREATE SCHEMA IF NOT EXISTS bronze
    MANAGED LOCATION 'abfss://eccomerce-container@ecommerceextdatalake.dfs.core.windows.net/bronze';

CREATE SCHEMA IF NOT EXISTS silver
    MANAGED LOCATION 'abfss://eccomerce-container@ecommerceextdatalake.dfs.core.windows.net/silver';

CREATE SCHEMA IF NOT EXISTS gold
    MANAGED LOCATION 'abfss://eccomerce-container@ecommerceextdatalake.dfs.core.windows.net/gold';


In [0]:
%sql
SHOW SCHEMAS;

databaseName
bronze
default
gold
information_schema
landing
silver
silver_raw


%md
## Create Volume

In [0]:
%sql
USE CATALOG ecommerce;
USE SCHEMA landing;

CREATE EXTERNAL VOLUME IF NOT EXISTS ecommerce_data
    LOCATION 'abfss://eccomerce-container@ecommerceextdatalake.dfs.core.windows.net/landing/data';

In [0]:
%fs ls /Volumes/ecommerce/landing/ecommerce_data

path,name,size,modificationTime
dbfs:/Volumes/ecommerce/landing/ecommerce_data/checkpoints/,checkpoints/,0,1784853503000
dbfs:/Volumes/ecommerce/landing/ecommerce_data/customers/,customers/,0,1784850210000
dbfs:/Volumes/ecommerce/landing/ecommerce_data/orders/,orders/,0,1784850226000
dbfs:/Volumes/ecommerce/landing/ecommerce_data/payments/,payments/,0,1784850222000
dbfs:/Volumes/ecommerce/landing/ecommerce_data/products/,products/,0,1784850218000
dbfs:/Volumes/ecommerce/landing/ecommerce_data/returns/,returns/,0,1784850232000
dbfs:/Volumes/ecommerce/landing/ecommerce_data/schemas/,schemas/,0,1784853487000


### Create meta data colume for schema and checkpoints

In [0]:
%sql
CREATE EXTERNAL VOLUME IF NOT EXISTS ecommerce.landing.autoloader_meta
    LOCATION 'abfss://eccomerce-container@ecommerceextdatalake.dfs.core.windows.net/landing/autoloader_meta';